In [1]:
import requests
import os
import json
import pandas as pd
import urllib.parse
from sqlalchemy import create_engine
from groq import Groq

In [2]:
# ==========================================
# 1. CONFIGURAÇÕES
# ==========================================
# Insira sua chave da API do Groq aqui
os.environ["GROQ_API_KEY"] = "<COLE A SUA CHAVE DA API DO GROQ AQUI>" 

# URL da API REST
API_URL = "http://127.0.0.1:5000/api/usuarios"

# Lê as credenciais do banco para salvar as mensagens posteriormente
with open('db_config.json', 'r') as f:
    DB_CONFIG = json.load(f)

client = Groq()
MODELO_GROQ = "llama-3.1-8b-instant"

In [3]:
# ==========================================
# 2. FUNÇÕES
# ==========================================
def obter_conexao():
    """Cria a engine de conexão com o SQL Server para salvar as mensagens."""
    senha_codificada = urllib.parse.quote_plus(DB_CONFIG['password'])
    string_conexao = (
        f"mssql+pyodbc://{DB_CONFIG['user']}:{senha_codificada}"
        f"@{DB_CONFIG['server']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
        "?driver=ODBC+Driver+17+for+SQL+Server"
    )
    return create_engine(string_conexao)

def obter_usuarios():
    try:
        response = requests.get(API_URL)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Erro ao conectar na API local: {e}")
        return []

def gerar_mensagem_marketing(nome_usuario):
    prompt_sistema = (
        "Você é um especialista em marketing bancário altamente persuasivo, cordial e moderno. "
        "Sua missão é criar mensagens diretas e personalizadas para clientes, "
        "destacando a importância de começar a investir o quanto antes para proteger o patrimônio "
        "e garantir a liberdade financeira. "
        "A mensagem DEVE ser em português do Brasil, ter um tom encorajador e conter no máximo 3 frases."
    )
    
    prompt_usuario = f"Crie a mensagem de marketing para o cliente chamado {nome_usuario}."

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": prompt_usuario}
            ],
            model=MODELO_GROQ,
            temperature=0.7,
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Erro ao gerar mensagem na IA: {e}"

In [4]:
# ==========================================
# 3. EXECUÇÃO PRINCIPAL
# ==========================================
def main():
    print("Buscando usuários na API local...\n")
    usuarios = obter_usuarios()
    
    if not usuarios:
        print("Nenhum usuário encontrado. Verifique se a API Flask está rodando.")
        return

    print(f"Total de {len(usuarios)} usuários encontrados. Gerando campanhas com IA...\n")
    print("=" * 70)
    
    # Lista para armazenar os resultados antes de enviar ao banco
    dados_para_banco = []
    
    for usuario in usuarios:
        nome = usuario.get("nome")
        id_usuario = usuario.get("id")
        
        mensagem_ia = gerar_mensagem_marketing(nome)
        
        print(f"👤 Cliente: {nome} (ID: {id_usuario})")
        print(f"📩 Mensagem:\n{mensagem_ia}")
        print("-" * 70)
        
        dados_para_banco.append({
            "id_usuario": id_usuario,
            "mensagem": mensagem_ia
        })

    # ==========================================
    # 4. SALVANDO NO SQL SERVER
    # ==========================================
    if dados_para_banco:
        print("\nSalvando mensagens geradas no SQL Server...")
        try:
            # Transforma a lista de dicionários em um DataFrame do Pandas
            df_mensagens = pd.DataFrame(dados_para_banco)
            
            # Conecta e salva na tabela 'tb_mensagens_marketing'
            engine = obter_conexao()
            
            # O if_exists='append' adiciona novas linhas a cada execução, 
            # Caso a tabela não exista, a mesma será criada
            df_mensagens.to_sql('tb_mensagens_marketing', con=engine, if_exists='append', index=False)
            
            print("✅ Sucesso! Todas as mensagens foram gravadas na tabela 'tb_mensagens_marketing'.")
        except Exception as e:
            print(f"❌ Erro ao salvar no banco de dados: {e}")

In [5]:
# Executa o processo
main()

Buscando usuários na API local...

Total de 5 usuários encontrados. Gerando campanhas com IA...

👤 Cliente: Linus Torvals (ID: 1)
📩 Mensagem:
"Olá, Linus! É hora de criar riqueza a longo prazo e garantir sua liberdade financeira. Investir agora pode ser a melhor decisão que você vai tomar para o futuro de sua família. Vamos criar um plano personalizado para ajudá-lo a alcançar seus sonhos!"
----------------------------------------------------------------------
👤 Cliente: Richard Stallman (ID: 2)
📩 Mensagem:
"Olá Richard, você está pronto para tomar o controle de seu futuro financeiro? Comece a investir hoje e proteja seu patrimônio para que você possa viver a vida que sempre sonhou, sem preocupações financeiras!"
----------------------------------------------------------------------
👤 Cliente: Guido van Rossum (ID: 3)
📩 Mensagem:
"Olá Guido,

Você é o criador do Python, um dos linguagens de programação mais poderosas do mundo. Mas, sabia que você também pode criar riqueza e liberdade f